# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
# YOUR CODE HERE
!pip install -q -U bitsandbytes transformers accelerate

In [2]:
# YOUR CODE HERE
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
# YOUR CODE HERE
test_query = "My order hasn't arrived yet and it's been 4 days. When will I get it?"
ground_truth = "Domestic orders deliver within 3-7 business days."

messages = [
    {"role": "system", "content": "You are a helpful customer support assistant."},
    {"role": "user", "content": test_query}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate deterministic baseline output
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=120,
    do_sample=False,
    temperature=None,
    top_p=None
)

# Decode only the newly generated response
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
baseline_output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Query:", test_query)
print("\nBaseline Output:\n", baseline_output)

Query: My order hasn't arrived yet and it's been 4 days. When will I get it?

Baseline Output:
 I'm sorry to hear that your order has not arrived yet. Please check the status of your order on our website or app, as we may have received an update from you regarding its delivery. If you haven't received any updates in this time frame, please contact us at [contact information] for further assistance. We appreciate your patience and understanding during this time.


---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
# YOUR CODE HERE
print("=== BASELINE EVALUATION ===")
print(f"Ground Truth Rule: {ground_truth}\n")
print(f"Model Response:\n{baseline_output}\n")

# Document hallucinations / inaccuracies
hallucination_notes = (
    "The base model lacks access to internal company SOPs. Instead of stating the exact "
    "3-7 business days domestic delivery window, it gives generic advice, asks for order/tracking "
    "numbers it cannot actually look up, or invents standard shipping timelines."
)
print("Assessment:", hallucination_notes)

=== BASELINE EVALUATION ===
Ground Truth Rule: Domestic orders deliver within 3-7 business days.

Model Response:
I'm sorry to hear that your order has not arrived yet. Please check the status of your order on our website or app, as we may have received an update from you regarding its delivery. If you haven't received any updates in this time frame, please contact us at [contact information] for further assistance. We appreciate your patience and understanding during this time.

Assessment: The base model lacks access to internal company SOPs. Instead of stating the exact 3-7 business days domestic delivery window, it gives generic advice, asks for order/tracking numbers it cannot actually look up, or invents standard shipping timelines.


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**